# Web pages as text

> Browser steps from Mind2Web as sysone decisions, in JevForge's ready-made records: which of 12 page elements to act on, on websites the head never saw. Four frozen encoders under the same head, two text encoders and the multimodal application's two encoders reading text alone, and GLiNER2's classifier zero-shot.

- skip_exec: true
- skip_showdoc: true

A browser agent works one step at a time: given the task and the page, which element does it act on next? [Mind2Web](https://huggingface.co/datasets/osunlp/Mind2Web) (CC-BY-4.0) recorded people doing tasks on real websites, step by step, and [JevForge-Mind2Web](https://huggingface.co/datasets/AndeyTait/JevForge-Mind2Web) (CC-BY-4.0) turns 7,014 of those steps into Jev typed decisions:

- the **state** is the page as text: the task, the operation the person performed (click, type, select, hover), and 12 candidate elements, each described by its tag and attributes;
- **`action`** is a *choice*: which of the 12 elements to act on (usually one is right);
- **`is_target`** is a *noul*: a statement that one of the 12, sometimes the right one and more often not, is the element to act on.

Its splits keep websites apart: the heads here train on steps from 49 websites and are scored on 8 others (the test split), and on 4 more (the `ood` split). That's the question that matters for an agent: how well does it do on a site it has never seen?

The question for sysone is which encoder to put under the head for this kind of decision. Four candidates, each frozen, with a head trained on its cached anchors:

- **ModernBERT-large**, the text application's default;
- **GLiNER2.5-Decide**'s encoder, the best frozen encoder on typed-decisions (the [GLiNER2.5-Decide tutorial](gliner_decide.ipynb));
- **NeoMME-260M** and **ModernVBERT**, the multimodal application's two encoders, here reading the page as text alone. Whether a screenshot helps them is the [screens tutorial](web_screens.ipynb)'s question; this one measures how well they read the text they'd get beside it.

::: {.callout-tip title="Jargon"}
- **Element, candidate**: a piece of the page an agent can act on, such as a link, a button or a text field. Each step here offers 12 candidates, one of them (rarely two) right.
- **Held-out websites**: websites none of whose steps were used for training. Doing well on them means the head learned something about pages in general, not about particular sites.
- **Frozen encoder, cached anchors**: only the head trains, so each anchor's vector is computed once and stored, and training reads the stored vectors.
- **Standardised anchors**: vectors shifted and scaled, channel by channel, to mean 0 and variance 1 on the training rows. The multimodal application does this by default, because its encoders' vectors put most of their length in one channel (the [comparison tutorial](neomme_vs_modernvbert.ipynb#the-geometry-of-the-anchors)).
- **Brier score**: the mean squared distance between the predicted probabilities and the gold distribution. Lower is better, and it rewards confidence only where it's right.
:::

::: {.callout-note title="Running this notebook"}
The test suite skips this notebook (`skip_exec`). It downloads JevForge-Mind2Web (30 MB), and uses ModernBERT-large (1.6 GB), GLiNER2.5-Decide (1.95 GB), NeoMME-260M (526 MB) and ModernVBERT (1.2 GB). One encoder is in memory at a time. The whole run takes about an hour on an M3 Pro (MPS, fp32), 36 minutes of it in GLiNER2.5-Decide's encoder, and the outputs below come from a run on 2026-09-30. It keeps its feature cache in `web-text/`, next to the notebook.
:::

In [ ]:
import gc, json, logging, random, resource, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import datasets, transformers
from huggingface_hub import snapshot_download
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from sysone.data import TypedDecisions, as_cases
import sysone.text as text_app
import sysone.multimodal as multimodal_app

In [ ]:
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*torch\.jit\.script", category=FutureWarning)   # DeBERTa-v2's modeling code on Python 3.14
datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 80)
DECIDE = "fastino/GLiNER2.5-Decide"
work = Path("web-text")
t_start = time.time()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"torch {torch.__version__} · transformers {transformers.__version__} · {device}")

torch 2.14.0 · transformers 5.17.0 · mps


In [ ]:
def quiet(learn):
    "The learner, without the Trainer's per-step log lines in the outputs"
    for cb in (PrinterCallback, ProgressCallback): learn.trainer.remove_callback(cb)
    return learn

def release():
    "Hand back the memory of deleted models"
    gc.collect()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def memory_line():
    "The process's peak resident memory"
    return f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**30 if sys.platform == 'darwin' else 2**20):.1f} GB"

def scores(learn, split, calibrated=True) -> dict:
    "Accuracy on each question, and the Brier score, on a held-out split"
    m = learn.validate(split, calibrated=calibrated)
    return {"element": m["accuracy_choice"], "is it the target?": m["accuracy_noul"], "Brier": m["brier"]}

## The records

A JevForge record is close to sysone's shape already: a request with the state and the questions, and the gold both as a label and as a distribution over the options.

In [ ]:
repo = Path(snapshot_download("AndeyTait/JevForge-Mind2Web", repo_type="dataset"))
raw = {s: [json.loads(l) for l in (repo / f"data/{s}.jsonl").read_text().splitlines()] for s in ("train", "calibration", "test", "ood")}
r = raw["test"][0]
st = json.loads(r["request"]["state"])
print(f"task: {st['task']}")
print(f"operation: {st['operation']} · website: {st['website']}")
for e in st["elements"]: print(f"{e['id']:>5}  {e['text']}")
{"questions": {q: {"type": v["type"], "instructions": v["instructions"]} for q, v in r["request"]["questions"].items()}, "gold": r["gold"]}

task: rent a car in Brooklyn - Central, NY on from April 9 to April 15.
operation: CLICK · website: united
   e1  div
   e2  li id=bookCarTab role=tab aria_label=heading level 3 Search and reserve a car
   e3  div
   e4  a title=Open United's LinkedIn profile in a new tab
   e5  div
   e6  li
   e7  h2 id=favoritesModalHeader
   e8  svg role=img aria_label=Save to favorites
   e9  svg role=img aria_label=Save to favorites
  e10  button role=link type=button
  e11  li
  e12  form id=mapSearchForm


{'questions': {'action': {'type': 'choice',
   'instructions': 'Which page element should be interacted with next to make progress on the task?'},
  'is_target': {'type': 'noul',
   'instructions': "The page element described as 'a title=Open United's LinkedIn profile in a new tab' is a correct next interaction target for completing the task."}},
 'gold': {'action': 'e2', 'is_target': False}}

The state arrives as a JSON string, and the gold as a label beside a distribution. `to_record` parses the state and pairs each question's label with its distribution, which is what sysone reads as a gold. A step with two right elements then has its target split between them.

In [ ]:
def to_record(r) -> dict:
    "A JevForge record in sysone's shape: the state, the questions, and each question's gold as a label and a distribution"
    gold = {q: {"label": str(g).lower() if isinstance(g, bool) else g, "probabilities": r["targets"][q]} for q, g in r["gold"].items()}
    return {"id": r["id"], "website": r["source_group"], "state": json.loads(r["request"]["state"]),
            "questions": r["request"]["questions"], "gold": gold}

records = {s: [to_record(r) for r in rs] for s, rs in raw.items()}
pd.DataFrame({s: {"steps": len(rs), "websites": len({r["website"] for r in rs}),
                  "share where the noul is true": round(np.mean([r["gold"]["is_target"]["label"] == "true" for r in rs]), 2)}
              for s, rs in records.items()}).T

,steps,websites,share where the noul is true
train,4642.0,49.0,0.21
calibration,400.0,6.0,0.26
test,800.0,8.0,0.22
ood,386.0,4.0,0.23


The heads train on 2,000 of the 4,642 training steps, drawn at random. That's 4,000 decisions, plenty for a head on cached anchors, and it keeps the slowest encoder's run under an hour. The calibration split fits the temperatures, and the test and `ood` splits are scored. Every encoder gets the same records.

In [ ]:
train = random.Random(0).sample(records["train"], 2000)

def make_data() -> TypedDecisions:
    "The same splits for every encoder: the training steps, the calibration split, and the two held-out splits"
    data = TypedDecisions(train, valid=records["test"], calib=records["calibration"])
    data.cases["ood"] = as_cases(records["ood"])            # a second held-out split, scored with learn.validate("ood")
    return data

print(f"{len({r['website'] for r in train})} websites in the training steps")

49 websites in the training steps


## One row

Each question becomes one row. Here are the two rows of the first test step as the GLiNER2.5-Decide encoder reads them, in its `gliner` template: the question type, the instructions, an `[L]` anchor before each option, then the page.

In [ ]:
data = make_data()
torch.manual_seed(0)
learn = quiet(text_app.decision_learner(data, DECIDE, head="gliner2", init_from=DECIDE, cache_dir=work))
learn.show_batch(2, split="valid", max_chars=360)

([P] choice: Which page element should be interacted with next to make progress on the task? ([L] e1: div[L] e2: li id=bookCarTab role=tab aria_label=heading level 3 Search and reserve a car[L] e3: div[L] e4: a title=Open United's LinkedIn  …  "operation": "CLICK", "task": "rent a car in Brooklyn - Central, NY on from April 9 to April 15.", "website": "united"}
markers [20, 25, 51, 56, 73, 78, 83, 96, 112, 128, 139, 144] · qtype choice · 508 tokens

([P] noul: The page element described as 'a title=Open United's LinkedIn profile in a new tab' is a correct next interaction target for completing the task. ([L] false: The element is not a correct next target.[L] true: The element is a cor …  "operation": "CLICK", "task": "rent a car in Brooklyn - Central, NY on from April 9 to April 15.", "website": "united"}
markers [37, 49] · qtype noul · 417 tokens



## Zero-shot: GLiNER2's own classifier

With GLiNER2's classifier copied into the head, the Decide encoder answers before any training, as the [GLiNER2.5-Decide tutorial](gliner_decide.ipynb) showed on Fastino's own benchmark. The scores use no temperatures, since nothing is fitted:

In [ ]:
t = time.time()
results = {"GLiNER2.5-Decide, its classifier, zero-shot": {s: scores(learn, s, calibrated=False) for s in ("valid", "ood")}}
print(f"the test and ood steps encoded and scored in {(time.time() - t) / 60:.1f} min")
del learn, data; release()
results

the test and ood steps encoded and scored in 11.9 min


{'GLiNER2.5-Decide, its classifier, zero-shot': {'valid': {'element': 0.24375,
   'is it the target?': 0.38625,
   'Brier': 0.695402393972732},
  'ood': {'element': 0.3005181347150259,
   'is it the target?': 0.31088082901554404,
   'Brier': 0.6933096424542362}}}

With no training at all, GLiNER2's classifier picks the right element on 24% of the test steps and 30% of the `ood` ones, three to four times better than guessing among 12 (8%). On the noul it does worse than always answering false, 39% and 31% right against 78% and 77%: it calls too many elements the target. Its scores come without fitted temperatures, and its Brier score shows it.

## Four encoders, one recipe

Each encoder runs through its own application with that application's defaults. The text application reads the anchors as they are and caches them in float16. The multimodal application standardises them and caches them in float32, for the reasons its notebooks give. The head is Laya's scorer in both, and every head trains the same way: `fit_one_cycle(4, lr=1e-3)`, then temperatures fitted on the calibration split. The [GLiNER2.5-Decide tutorial](gliner_decide.ipynb#how-fast-to-train) explains why the rate is fixed rather than picked by `lr_find`.

In [ ]:
ENCODERS = {"ModernBERT-large": (text_app, "answerdotai/ModernBERT-large"), "GLiNER2.5-Decide": (text_app, DECIDE),
            "NeoMME-260M": (multimodal_app, "Hcompany/NeoMME-260M"), "ModernVBERT": (multimodal_app, "ModernVBERT/modernvbert")}
minutes = {}
for name, (app, encoder) in ENCODERS.items():
    t = time.time()
    torch.manual_seed(0)                                     # the same starting head for every encoder, and on every run
    learn = quiet(app.decision_learner(make_data(), encoder, cache_dir=work))
    learn.fit_one_cycle(4, lr=1e-3)                          # the temperatures are fitted on the calibration split after it
    results[f"{name}, a new head"] = {s: scores(learn, s) for s in ("valid", "ood")}
    minutes[name] = (time.time() - t) / 60
    print(f"{name}: {minutes[name]:.1f} min · {memory_line()}")
    del learn; release()

ModernBERT-large: 10.7 min · peak memory 4.0 GB
GLiNER2.5-Decide: 24.2 min · peak memory 4.5 GB
NeoMME-260M: 7.5 min · peak memory 4.5 GB
ModernVBERT: 4.8 min · peak memory 4.5 GB


## Results

Accuracy on the element choice and on the noul, and the Brier score, on the 8 test websites and on the 4 `ood` ones. The first row is what guessing gets: one element in 12, and *false* for every noul, since most nouls are about a wrong element.

In [ ]:
guess = {s: {"element": 1 / 12, "is it the target?": np.mean([r["gold"]["is_target"]["label"] == "false" for r in records[k]]), "Brier": np.nan}
         for s, k in (("valid", "test"), ("ood", "ood"))}
rows = {"guessing": guess} | results
table = pd.concat({label: pd.DataFrame({name: r[split] for name, r in rows.items()}).T
                   for split, label in (("valid", "test websites"), ("ood", "ood websites"))}, axis=1)
table.style.format("{:.3f}", na_rep="")

- **The four encoders are close on the element.** On the test websites the new heads score between 0.374 and 0.406, and on the `ood` ones between 0.399 and 0.446. One standard error is about 0.017 on 800 test steps and 0.025 on 386 `ood` steps, so only the widest gap stands out: NeoMME-260M (0.406 and 0.446) against ModernBERT-large (0.374 and 0.399). Every head is about five times better than guessing, and far from solving the task.
- **GLiNER2.5-Decide's lead doesn't carry over.** It was the best frozen encoder on typed-decisions (0.645) and on Fastino's own benchmark, but here it sits in the middle, at 0.393 and 0.417. Its template also has less room: DeBERTa-v3 reads 512 positions, and the `gliner` template writes these rows longer than Laya's does, so 7% of the test rows have their page cut to fit, against under 1% for ModernBERT-large.
- **The noul stays near the prior.** About one statement in five is true, and no head moves far from always answering false: between 0.766 and 0.806 on the test websites, against 0.776. A head on frozen anchors learns little about one named element from 2,000 examples. The element choice, which weighs all 12 at once, is where the heads learn something.
- **The multimodal encoders read web text well.** NeoMME-260M, reading the page as text with its application's defaults, has the best element accuracy on the test websites and ties ModernVBERT on the `ood` ones, and it has the lowest Brier score on both.

In [ ]:
pd.Series(minutes, name="minutes, encoding and training included").round(1).to_frame()

,"minutes, encoding and training included"
ModernBERT-large,10.7
GLiNER2.5-Decide,24.2
NeoMME-260M,7.5
ModernVBERT,4.8


GLiNER2.5-Decide's encoder is by far the slowest: 24 minutes for the training and calibration rows, 36 in all with the zero-shot cell's encoding of the test rows, where NeoMME-260M took 7.5 minutes and ModernVBERT 4.8 for everything. DeBERTa-v3-large is the largest of the four, and it has no fused attention kernel in transformers, so it runs the slower eager path.

## What this shows

- **On web pages as text, the encoder matters less than on generic decisions.** On typed-decisions the frozen encoders spread over 14 points, from 0.504 to 0.645 (the [text notebook](../10_text.ipynb#regimes-on-the-laptop)); here the trained heads spread over 3 to 5.
- **NeoMME-260M is the best frozen encoder here, and one of the cheapest.** It leads on the test websites, ties on the `ood` ones, and costs 7.5 minutes for the whole run. For the multimodal application that means its text side is no weakness; whether its image side helps is the [screens tutorial](web_screens.ipynb)'s question.
- **GLiNER2.5-Decide's encoder isn't worth its cost for web pages.** It trails NeoMME by one to three points at nearly five times the encoding time. Its strength is decisions phrased as questions over ordinary text, where it leads by eight points or more.
- **What would move these numbers.** Training the encoder itself (`train="lora"`), or all 4,642 training steps. A frozen encoder with a head reaches about 0.40 on a 12-way element choice, and a web agent needs much more.

In [ ]:
print(f"the whole notebook: {(time.time() - t_start) / 60:.1f} minutes · {memory_line()}")

the whole notebook: 59.3 minutes · peak memory 4.5 GB
